# Preprocessing refinement

Step 1 repairs recorded line break artifacts. Step 2 scans the formal corpus for residual
markup and applies two narrow normalization rules. The final section describes the next
preprocessing notebook and subsequent analysis experiments.

## Step 1 · Repair recorded separator artifacts

Use the verified Notebook 03 structure and overlap evidence with the Notebook 04 first pass.
Repair extra newlines only when they intersect a recoverable separator inserted by Notebook 03.
Recompute paragraph and chunk offsets for changed records with the two Notebook 04 budgets
of 1,200 and 2,400 Unicode code points. These paragraph-based plans preserve paragraphs
that exceed their target. Model-specific limits depend on the selected tokenizer.

Run All in the project kernel. The shared preprocessing configuration defaults to
local[6]. Set FOT05_PROFILE to override the profile.
The compact result is stored under data/processed/preprocessing/formal_layout_step_v1. A
completed snapshot with the same source and rule is verified and reused. Set FOT05_OUTPUT to a
new directory inside preprocessing when changing the policy. Shut down the kernel after
inspecting the views.

In [ ]:
import ast
import hashlib
import json
import os
import re
import sys
import tempfile
import time
from pathlib import Path

import nbformat
import pyarrow.parquet as pq
from IPython import get_ipython
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "configs/preprocessing_notebook.json").is_file()
    ),
    None,
)
if ROOT is None or Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Open the project workspace and select its registered kernel.")

config = json.loads((ROOT / "configs/preprocessing_notebook.json").read_text())
profile = os.environ.get("FOT05_PROFILE", config["default_profile"])
resource = config["profiles"][profile]
STRUCTURE = ROOT / "work_packages/formal_corpus_release_20261117/structure-v2"
OVERLAP = STRUCTURE.with_name(f"{STRUCTURE.name}-overlap-v1")
output_root = (ROOT / config["output_directory"]).resolve()
FIRST_PASS = output_root / "formal_first_pass_v1"
OUTPUT = Path(os.environ.get("FOT05_OUTPUT", output_root / "formal_layout_step_v1")).resolve()
if OUTPUT.parent != output_root or OUTPUT == FIRST_PASS:
    raise ValueError("Use a separate Notebook 05 output directory under the preprocessing root.")

if SparkSession.getActiveSession() is not None:
    raise RuntimeError("Start Notebook 05 in a fresh kernel.")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS", f"--driver-memory {resource['driver_memory']} pyspark-shell"
)
spark = (
    SparkSession.builder.master(resource["master"])
    .appName("fear-temperature-layout-step")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", resource["shuffle_partitions"])
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024**2))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
if spark.version != config["spark_version"] or spark.sparkContext.master != resource["master"]:
    raise RuntimeError("Spark runtime differs from the selected project profile.")
print(f"Spark {spark.version} · {spark.sparkContext.master} · Notebook 05")

## Verify the rule on concrete cases

Project each Notebook 03 insertion into the SHA-bound Notebook 04 text. If Notebook 04 edited
through that separator, retain the text and flag the case. For a run of at least three line feeds
with an intact separator, replace the run with two line feeds and record the code-point offsets,
the original and replacement text, and the relevant Notebook 03 edit ordinal.

Preserve every non-newline character and the same paragraph boundaries. The changed records
must replay their prior Notebook 04 chunk plans before receiving new offsets.

In [ ]:
LAYOUT_RULE = "collapse_03_separator_newlines"
EXTRA_NEWLINES = re.compile(r"\n{3,}")
STRUCTURE_SEPARATOR = "\n\n"
PARAGRAPH_BREAK = re.compile(r"(?:\r?\n[ \t]*){2,}")


def text_sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def replay_edits(text, edits):
    cursor = 0
    pieces = []
    for start, end, _, before, replacement in edits:
        if not cursor <= start < end <= len(text) or text[start:end] != before:
            raise ValueError("An edit is out of order or its source slice does not match.")
        pieces.extend((text[cursor:start], replacement))
        cursor = end
    pieces.append(text[cursor:])
    return "".join(pieces)


def separator_positions(text03, structure_edits, edits04):
    """Project intact inserted separators onto the Notebook 04 text."""
    offsets = []
    blocked = 0
    shift03 = cursor03 = 0
    for index, (start, end, *_) in enumerate(structure_edits):
        if not cursor03 <= start < end:
            raise ValueError("Notebook 03 removal offsets are out of order.")
        cursor03 = end
        low = start + shift03
        high = low + len(STRUCTURE_SEPARATOR)
        if text03[low:high] != STRUCTURE_SEPARATOR:
            raise ValueError("Notebook 03 offsets do not identify the recorded separators.")
        shift03 += len(STRUCTURE_SEPARATOR) - (end - start)
        if any(lo < high and low < hi for lo, hi, *_ in edits04):
            blocked += 1
            continue
        shift04 = sum(
            len(replacement) - (hi - lo) for lo, hi, _, _, replacement in edits04 if hi <= low
        )
        offsets.append((index, low + shift04, high + shift04))
    return offsets, blocked


def paragraph_spans(text):
    spans = []
    start = 0
    for boundary in PARAGRAPH_BREAK.finditer(text):
        if text[start : boundary.start()].strip():
            spans.append((start, boundary.start()))
        start = boundary.end()
    if text[start:].strip():
        spans.append((start, len(text)))
    return spans


def paragraph_chunk_ends(length, spans, target):
    if not spans:
        return []
    ends = []
    chunk_start = previous_end = 0
    for unit_end in (*[span[0] for span in spans[1:]], length):
        if previous_end > chunk_start and unit_end - chunk_start > target:
            ends.append(previous_end)
            chunk_start = previous_end
        previous_end = unit_end
    ends.append(length)
    return ends


def project_after_layout(offset, edits):
    """Shift an offset on or outside a layout edit boundary."""
    shift = 0
    for start, end, _, _, replacement in edits:
        if start < offset < end:
            raise ValueError("A paragraph boundary falls inside a removed newline run.")
        if end <= offset:
            shift += len(replacement) - (end - start)
    return offset + shift


def repair_layout(text03, sha03, structure_edits, body04, sha04, edits04, inherited_plans):
    if text_sha(text03) != sha03:
        raise ValueError("Notebook 03 body hash mismatch.")
    text04 = replay_edits(text03, edits04)
    if (body04 is not None and body04 != text04) or (edits04 and body04 is None):
        raise ValueError("Notebook 04 override disagrees with its edit ledger.")
    if text_sha(text04) != sha04:
        raise ValueError("Notebook 04 body hash mismatch.")
    positions, blocked = separator_positions(text03, structure_edits, edits04)
    for _, start, end in positions:
        if text04[start:end] != STRUCTURE_SEPARATOR:
            raise ValueError("A projected separator disagrees with the Notebook 04 text.")
    runs = list(EXTRA_NEWLINES.finditer(text04))
    edits = []
    proof = []
    boundary_edits = 0
    for run in runs:
        start, end = run.span()
        matching = [(i, low, high) for i, low, high in positions if low < end and start < high]
        if matching:
            edits.append((start, end, LAYOUT_RULE, run.group(), STRUCTURE_SEPARATOR))
            proof.extend((start, end, i, low, high) for i, low, high in matching)
            boundary_edits += int(start == 0 or end == len(text04))

    review = []
    if blocked:
        review.append("upstream_separator_changed_in_04")
    remaining = len(runs) - len(edits)
    if remaining:
        review.append("extra_newline_run_without_intact_03_separator")
    if not edits:
        return sha04, len(text04), len(runs), [], [], boundary_edits, None, None, None, review

    cleaned = replay_edits(text04, edits)
    before_spans = paragraph_spans(text04)
    after_spans = paragraph_spans(cleaned)
    expected_spans = [
        (project_after_layout(start, edits), project_after_layout(end, edits))
        for start, end in before_spans
    ]
    if expected_spans != after_spans:
        raise ValueError("The newline rule changed the paragraph boundaries.")
    plans = []
    for target, previous_ends in inherited_plans:
        if paragraph_chunk_ends(len(text04), before_spans, target) != previous_ends:
            raise ValueError("The Notebook 04 chunk policy did not replay on a changed record.")
        ends = paragraph_chunk_ends(len(cleaned), after_spans, target)
        if after_spans:
            valid = (
                bool(ends)
                and ends[-1] == len(cleaned)
                and all(lower < upper for lower, upper in zip([0, *ends[:-1]], ends, strict=True))
            )
        else:
            valid = not ends
        if not valid:
            raise ValueError("The revised plan does not cover the complete cleaned body.")
        plans.append((target, previous_ends, ends))
    return (
        text_sha(cleaned),
        len(cleaned),
        len(runs),
        edits,
        proof,
        boundary_edits,
        cleaned,
        after_spans,
        plans,
        review,
    )

In [ ]:
targets = (1200, 2400)
separator_edits = [(4, 12, "test_upstream_removal", "test_sha")]


def layout_case(text, removals, first_edits=()):
    source04 = replay_edits(text, first_edits)
    starts = paragraph_spans(source04)
    plans = [(target, paragraph_chunk_ends(len(source04), starts, target)) for target in targets]
    return repair_layout(
        text,
        text_sha(text),
        removals,
        source04 if first_edits else None,
        text_sha(source04),
        first_edits,
        plans,
    )


examples = [
    ("Heat\n\n\n\n", separator_edits, (), "Heat\n\n", 1, 1),
    ("Heat\n\n\nStill hot.", separator_edits, (), "Heat\n\nStill hot.", 1, 0),
    ("Heat\n\nStill hot.", separator_edits, (), None, 0, 0),
    ("Heat\n\n\nStill hot.", [], (), None, 0, 0),
    (
        "Heat\n\n\nStill hot.",
        separator_edits,
        [(0, 4, "test", "Heat", "Too hot")],
        "Too hot\n\nStill hot.",
        1,
        0,
    ),
    ("Heat\n\n\nStill hot.", separator_edits, [(4, 5, "test", "\n", "")], None, 0, 0),
    (
        "Heat\n\n\nStill not afraid at 40°C 👩🏽‍🚒.",
        separator_edits,
        (),
        "Heat\n\nStill not afraid at 40°C 👩🏽‍🚒.",
        1,
        0,
    ),
    (
        "x" * 1194 + "\n\n\n" + "safe",
        [(1194, 1200, "test_upstream_removal", "test_sha")],
        (),
        "x" * 1194 + "\n\n" + "safe",
        1,
        0,
    ),
    (
        "Heat\n\n\nFine\n\n\nAgain.",
        [(4, 12, "test", "sha_one"), (17, 21, "test", "sha_two")],
        (),
        "Heat\n\nFine\n\nAgain.",
        2,
        0,
    ),
    ("\n\n\n", [(0, 3, "test", "sha")], (), "\n\n", 1, 1),
]
for index, (text, removals, first_edits, expected, count, boundary) in enumerate(examples):
    observed = layout_case(text, removals, first_edits)
    assert observed[6] == expected, (index, observed[6], expected)
    assert len(observed[3]) == count and observed[5] == boundary
    if expected is not None:
        before = replay_edits(text, first_edits)
        assert replay_edits(before, observed[3]) == expected
        assert observed[0] == text_sha(expected)
        assert re.sub(r"\n+", "\n", before) == re.sub(r"\n+", "\n", expected)
assert "upstream_separator_changed_in_04" in layout_case(*examples[5][:3])[-1]
assert "extra_newline_run_without_intact_03_separator" in layout_case(*examples[3][:3])[-1]
changed_plan = layout_case(*examples[7][:3])[8][0]
assert (len(changed_plan[1]), len(changed_plan[2])) == (2, 1)
try:
    repair_layout("Heat", "invalid", [], None, text_sha("Heat"), [], [])
except ValueError as exc:
    assert "Notebook 03 body hash" in str(exc)
else:
    raise AssertionError("An unbound source body was accepted.")
print(f"{len(examples)} layout cases and one rejected hash case passed.")

## Run the bounded rule on the formal release

Verify the formal parent inventory and the parts used by this step. The rule considers every
parent with a recorded Notebook 03 removal span. The compact evidence table keeps one row for
each of these parents, with a text override and recomputed plans when a change is applied.
The published inputs provide the remaining parents.

In [ ]:
def file_sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def verify_parts(folder, tables):
    checked = 0
    for name, entries in tables.items():
        table = folder / name
        files = {path.relative_to(folder).as_posix(): path for path in table.glob("*.parquet")}
        expected = {entry["path"]: entry for entry in entries}
        if not files or set(files) != set(expected) or not (table / "_SUCCESS").is_file():
            raise ValueError(f"Unexpected inventory for {table.relative_to(ROOT)}.")
        for name, path in files.items():
            entry = expected[name]
            if (
                file_sha(path) != entry["sha256"]
                or pq.ParquetFile(path).metadata.num_rows != entry["rows"]
            ):
                raise ValueError(f"Parquet hash or row count mismatch in {name}.")
        checked += len(files)
    return checked


def read_snapshot(folder, audit_filename, audit_field):
    manifest_path = folder / "MANIFEST.json"
    audit_path = folder / audit_filename
    manifest = json.loads(manifest_path.read_text())
    audit = json.loads(audit_path.read_text())
    manifest_sha = file_sha(manifest_path)
    valid_hash = (
        audit[audit_field] == manifest_sha
        if audit_field == "manifest_sha256"
        else manifest[audit_field] == file_sha(audit_path)
    )
    if not valid_hash:
        raise ValueError(f"The manifest and audit disagree in {folder.name}.")
    return manifest, audit, manifest_sha


structure_manifest, structure_audit, structure_sha = read_snapshot(
    STRUCTURE, "STRUCTURE_AUDIT.json", "manifest_sha256"
)
first_manifest, first_audit, first_sha = read_snapshot(FIRST_PASS, "AUDIT.json", "audit_sha256")
overlap_manifest, overlap_audit, overlap_sha = read_snapshot(OVERLAP, "AUDIT.json", "audit_sha256")
population = structure_audit["selected_records"]
split_counts = structure_audit["selected_split_counts"]
if (
    structure_manifest["run_mode"] != "full"
    or population != 718_330
    or first_audit["input_records"] != population
    or first_audit["split_counts"] != split_counts
    or first_manifest["input_structure_manifest_sha256"] != structure_sha
    or overlap_manifest["structure_manifest_sha256"] != structure_sha
    or overlap_audit["source_records"] != population
):
    raise ValueError("Notebook 03, Notebook 04 and the overlap evidence do not agree.")

structure_tables = {"paragraph_records": structure_manifest["tables"]["paragraph_records"]["files"]}
first_tables = {"records": first_manifest["records_files"]}
for inventory in (structure_tables["paragraph_records"], first_tables["records"]):
    if sum(entry["rows"] for entry in inventory) != population:
        raise ValueError("The verified input roster does not cover the formal population.")

verified = {
    "structure": verify_parts(STRUCTURE, structure_tables),
    "first_pass": verify_parts(FIRST_PASS, first_tables),
    "overlap": verify_parts(OVERLAP, overlap_manifest["tables"]),
}


# Verify the latest executed policy against its saved, output-independent source.
def canonical(source):
    return ast.dump(ast.parse(source), include_attributes=False)


notebook_path = Path(
    os.environ.get("FOT05_SOURCE_NOTEBOOK", ROOT / "notebooks/05_preprocessing_refinement.ipynb")
)
notebook = nbformat.read(notebook_path, as_version=4)
saved_policy = next(cell.source for cell in notebook.cells if cell.id == "05-layout-policy")
kernel = get_ipython()
latest = None
if kernel is not None:
    for source in reversed(kernel.history_manager.input_hist_raw):
        try:
            parsed = ast.parse(source)
        except SyntaxError:
            continue
        if any(
            isinstance(item, ast.FunctionDef) and item.name == "repair_layout"
            for item in parsed.body
        ):
            latest = canonical(source)
            break
if latest != canonical(saved_policy):
    raise ValueError("Save Notebook 05 and rerun the saved layout policy before publication.")

identity = {
    "schema_version": 1,
    "policy": LAYOUT_RULE,
    "policy_sha256": text_sha(latest),
    "input_structure_manifest_sha256": structure_sha,
    "input_first_pass_manifest_sha256": first_sha,
    "input_overlap_manifest_sha256": overlap_sha,
    "chunk_targets": first_manifest["chunk_targets"],
}
if list(targets) != identity["chunk_targets"]:
    raise ValueError("Regression targets disagree with the two published chunk plans.")

structure_rows = spark.read.parquet(str(STRUCTURE / "paragraph_records"))
first_rows = spark.read.parquet(str(FIRST_PASS / "records"))
overlap_rows = spark.read.parquet(str(OVERLAP / "records"))
observed_splits = {
    row.ml_split: row["count"] for row in first_rows.groupBy("ml_split").count().collect()
}
if observed_splits != split_counts:
    raise ValueError("The first-pass Parquet rows disagree with the formal split roster.")
print(
    f"Verified the {population:,}-record parent roster and "
    f"{sum(verified.values())} input Parquet files."
)

In [ ]:
payload = T.StructType.fromDDL("""
    output_sha256 STRING, output_chars INT, input_newline_runs INT,
    layout_edits ARRAY<STRUCT<start: INT, end: INT, rule: STRING,
                              before: STRING, replacement: STRING>>,
    separator_proof ARRAY<STRUCT<edit_start: INT, edit_end: INT, structure_edit_ordinal: INT,
                                 input_start: INT, input_end: INT>>,
    boundary_edits INT, clean_text_override STRING,
    output_paragraphs ARRAY<STRUCT<start: INT, end: INT>>,
    replanned ARRAY<STRUCT<target: INT, input_ends: ARRAY<INT>, output_ends: ARRAY<INT>>>,
    layout_review_flags ARRAY<STRING>
""")
keys = ("release_row", "native_identity", "release_group_id", "ml_split", "format", "source_host")
keys03 = (*keys, "structure_sha256")

if OUTPUT.is_dir():
    output_manifest, result_audit, _ = read_snapshot(OUTPUT, "AUDIT.json", "audit_sha256")
    if any(output_manifest.get(k) != v for k, v in identity.items()):
        raise ValueError("Choose a new FOT05_OUTPUT directory for a changed input or policy.")
    verify_parts(OUTPUT, output_manifest["tables"])
    print(f"Reusing verified layout snapshot {OUTPUT.relative_to(ROOT)}.")
else:
    started = time.monotonic()
    with tempfile.TemporaryDirectory(prefix=".formal-layout-", dir=output_root) as temp:
        staged = Path(temp) / "snapshot"
        sources = structure_rows.where(F.size("removed_spans_in_release_clean_text") > 0).select(
            *keys,
            F.col("derived_sha256").alias("structure_sha256"),
            "derived_text",
            "removed_spans_in_release_clean_text",
        )
        targets04 = first_rows.select(
            *keys,
            F.col("input_sha256").alias("structure_sha256"),
            F.col("clean_sha256").alias("input_sha256"),
            F.col("clean_chars").alias("input_chars"),
            "clean_text_override",
            "edits",
            "plans",
        )
        linked = targets04.join(F.broadcast(sources), list(keys03), "inner")
        repair = F.udf(repair_layout, payload, useArrow=False)
        (
            linked.select(
                *keys03,
                "input_sha256",
                "input_chars",
                repair(
                    "derived_text",
                    "structure_sha256",
                    "removed_spans_in_release_clean_text",
                    "clean_text_override",
                    "input_sha256",
                    "edits",
                    "plans",
                ).alias("layout"),
            )
            .select(*keys03, "input_sha256", "input_chars", "layout.*")
            .write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged / "records"))
        )
        result = spark.read.parquet(str(staged / "records"))
        result = result.withColumn("edits_applied", F.size("layout_edits"))
        counts = (
            result.agg(
                F.count("*").alias("records_with_03_edits_examined"),
                F.countDistinct("release_row").alias("unique_parent_rows"),
                F.countDistinct("native_identity").alias("unique_parent_identities"),
                F.count_if(F.col("edits_applied") > 0).alias("edited_records"),
                F.sum("input_newline_runs").alias("input_runs"),
                F.sum("edits_applied").alias("repaired_runs"),
                F.sum("boundary_edits").alias("boundary_runs_repaired"),
                F.sum(F.col("input_newline_runs") - F.col("edits_applied")).alias(
                    "remaining_runs_in_scope"
                ),
                F.sum(F.col("input_chars") - F.col("output_chars")).alias("net_codepoints_removed"),
                F.count_if(F.size("layout_review_flags") > 0).alias("review_records"),
                F.sum(
                    F.aggregate(
                        "layout_edits",
                        F.lit(0),
                        lambda total, edit: (
                            total + F.length(edit.before) - F.length(edit.replacement)
                        ),
                    )
                ).alias("net_codepoints_in_ledger"),
                F.count_if(
                    ((F.col("edits_applied") == 0) != F.col("clean_text_override").isNull())
                    | (
                        (F.col("edits_applied") > 0)
                        & (F.sha2("clean_text_override", 256) != F.col("output_sha256"))
                    )
                ).alias("bad_overrides"),
            )
            .first()
            .asDict()
        )
        expected = structure_audit["edited_records"]
        if (
            any(
                counts[name] != expected
                for name in (
                    "records_with_03_edits_examined",
                    "unique_parent_rows",
                    "unique_parent_identities",
                )
            )
            or counts["bad_overrides"] != 0
            or counts["net_codepoints_removed"] != counts["net_codepoints_in_ledger"]
        ):
            raise ValueError(f"Layout record coverage or body verification failed: {counts}")
        by_split = [
            row.asDict()
            for row in result.groupBy("ml_split")
            .agg(
                F.count("*").alias("examined"),
                F.count_if(F.col("edits_applied") > 0).alias("edited"),
            )
            .orderBy("ml_split")
            .collect()
        ]
        chunk_deltas = (
            result.select(F.explode("replanned").alias("plan"))
            .select("plan.*")
            .groupBy("target")
            .agg(
                F.count("*").alias("replanned_records"),
                F.count_if(F.size("input_ends") != F.size("output_ends")).alias(
                    "records_with_chunk_count_change"
                ),
                F.sum(F.size("output_ends") - F.size("input_ends")).alias("chunk_count_delta"),
            )
            .orderBy("target")
            .collect()
        )
        if [row.target for row in chunk_deltas] != list(targets) or any(
            row.replanned_records != counts["edited_records"] for row in chunk_deltas
        ):
            raise ValueError("Each edited record must replay the two Notebook 04 plans.")
        input_chunk_totals = {
            row["target"]: row["chunks"]
            for row in first_audit["chunk_statistics"]
            if row["section"] == "overall"
        }
        result_audit = {
            "formal_records": population,
            "split_counts": split_counts,
            "scope": "All parents with recorded Notebook 03 structural edit spans",
            "counts": counts,
            "by_split": by_split,
            "chunk_summary": [
                {
                    **row.asDict(),
                    "input_chunks": input_chunk_totals[row.target],
                    "output_chunks": input_chunk_totals[row.target] + row.chunk_count_delta,
                }
                for row in chunk_deltas
            ],
            "elapsed_seconds": round(time.monotonic() - started, 1),
        }
        audit_path = staged / "AUDIT.json"
        audit_path.write_text(json.dumps(result_audit, indent=2) + "\n")
        files = [
            {
                "path": path.relative_to(staged).as_posix(),
                "sha256": file_sha(path),
                "rows": pq.ParquetFile(path).metadata.num_rows,
            }
            for path in sorted((staged / "records").glob("*.parquet"))
        ]
        output_manifest = {
            **identity,
            "source_parts_verified": verified,
            "audit_sha256": file_sha(audit_path),
            "spark_version": spark.version,
            "spark_master": spark.sparkContext.master,
            "tables": {"records": files},
            "records_schema": result.drop("edits_applied").schema.jsonValue(),
        }
        (staged / "MANIFEST.json").write_text(json.dumps(output_manifest, indent=2) + "\n")
        verify_parts(staged, output_manifest["tables"])
        for folder, expected_sha in (
            (STRUCTURE, structure_sha),
            (FIRST_PASS, first_sha),
            (OVERLAP, overlap_sha),
        ):
            if file_sha(folder / "MANIFEST.json") != expected_sha:
                raise ValueError("An input manifest changed during the layout run.")
        if OUTPUT.exists():
            raise FileExistsError("The layout destination appeared during publication.")
        staged.rename(OUTPUT)
        print(f"Published the layout snapshot at {OUTPUT.relative_to(ROOT)}.")

layout_evidence = spark.read.parquet(str(OUTPUT / "records"))

## Inspect the derived record view

The record view carries each parent's inherited split, the Notebook 04 review reasons and
the bound overlap evidence. The text view adds Notebook 03 review reasons and reconstructs
the final body from the ordered snapshots. Use a narrow parent filter for interactive
inspection.

In [ ]:
patch_keys = ("release_row", "native_identity", "first_pass_sha256")
related_keys = (
    "release_row",
    "native_identity",
    "release_group_id",
    "ml_split",
    "structure_sha256",
)


def layout_record_view(first, evidence, overlap):
    parents = (
        first.withColumnRenamed("clean_sha256", "first_pass_sha256")
        .withColumnRenamed("input_sha256", "structure_sha256")
        .withColumnRenamed("clean_text_override", "first_pass_override")
    )
    edits = evidence.select(
        "release_row",
        "native_identity",
        F.col("input_sha256").alias("first_pass_sha256"),
        "output_sha256",
        "output_chars",
        F.col("clean_text_override").alias("layout_override"),
        "layout_edits",
        F.col("output_paragraphs").alias("layout_paragraphs"),
        "replanned",
        "layout_review_flags",
        "input_newline_runs",
    )
    related = overlap.select(
        "release_row",
        "native_identity",
        "release_group_id",
        "ml_split",
        F.col("derived_sha256").alias("structure_sha256"),
        "overlap_status",
        "overlap_rules",
        "overlap_pair_ids",
    )
    return (
        parents.join(F.broadcast(edits), list(patch_keys), "left")
        .join(F.broadcast(related), list(related_keys), "left")
        .withColumns(
            {
                "clean_sha256": F.coalesce("output_sha256", "first_pass_sha256"),
                "clean_chars": F.coalesce("output_chars", "clean_chars"),
                "plans": F.coalesce(
                    F.transform(
                        "replanned",
                        lambda plan: F.struct(
                            plan.target.alias("target"), plan.output_ends.alias("ends")
                        ),
                    ),
                    "plans",
                ),
                "layout_status": F.when(F.col("layout_override").isNotNull(), "edited")
                .when(F.col("output_sha256").isNotNull(), "no_edit_for_separator_rule")
                .otherwise("outside_separator_rule_scope"),
                "layout_noop_reason": F.when(
                    F.col("output_sha256").isNotNull() & F.col("layout_override").isNull(),
                    F.when(F.col("input_newline_runs") == 0, "no_extra_newline_run").otherwise(
                        "no_run_with_intact_separator"
                    ),
                ),
                "overlap_status": F.coalesce(
                    "overlap_status", F.lit("no_cross_split_evidence_in_current_audit")
                ),
            }
        )
        .drop("output_sha256", "output_chars", "replanned")
    )


def layout_text_view(parents, structure):
    source = structure.select(
        "release_row",
        "native_identity",
        F.col("derived_sha256").alias("structure_sha256"),
        F.col("review_flags").alias("structure_review_reasons"),
        F.col("paragraphs_in_derived_text").alias("structure_paragraphs"),
        "derived_text",
    )
    return (
        parents.join(source, ["release_row", "native_identity", "structure_sha256"], "inner")
        .withColumn(
            "clean_text", F.coalesce("layout_override", "first_pass_override", "derived_text")
        )
        .drop("layout_override", "first_pass_override", "derived_text")
    )


layout_records = layout_record_view(first_rows, layout_evidence, overlap_rows)
layout_text_records = layout_text_view(layout_records, structure_rows)

# Use a record with a changed chunk count and an overlap-evidence control.
plan_changed = (
    layout_evidence.where(
        F.exists("replanned", lambda plan: F.size(plan.input_ends) != F.size(plan.output_ends))
    )
    .select("release_row")
    .first()
)
edited = layout_evidence.where(F.size("layout_edits") > 0).select("release_row").first()
control = (
    overlap_rows.where(F.col("overlap_status") == "direct_evidence").select("release_row").first()
)
chosen = {(plan_changed or edited).release_row, control.release_row}
selected = F.col("release_row").isin(*chosen)
probe = (
    layout_text_view(
        layout_record_view(
            first_rows.where(selected),
            layout_evidence.where(selected),
            overlap_rows.where(selected),
        ),
        F.broadcast(structure_rows.where(selected)),
    )
    .select(
        "release_row",
        "clean_text",
        "clean_sha256",
        "clean_chars",
        "plans",
        "layout_status",
        "overlap_status",
        "overlap_pair_ids",
        "ml_split",
    )
    .collect()
)
if {record.release_row for record in probe} != chosen:
    raise ValueError("The layout view does not return exactly one row per selected parent.")
for record in probe:
    if (
        text_sha(record.clean_text) != record.clean_sha256
        or len(record.clean_text) != record.clean_chars
    ):
        raise ValueError("The reconstructed layout text or length does not match its receipt.")
    for _, ends in record.plans:
        if not ends and not record.clean_text.strip():
            continue
        ranges = zip([0, *ends[:-1]], ends, strict=True)
        if "".join(record.clean_text[start:end] for start, end in ranges) != record.clean_text:
            raise ValueError("A rebuilt chunk plan does not cover its parent.")
if not any(row.layout_status == "edited" for row in probe):
    raise ValueError("The view check did not exercise a changed record.")
if not any(row.overlap_status == "direct_evidence" and row.overlap_pair_ids for row in probe):
    raise ValueError("The view check did not preserve the selected overlap evidence.")
print("Verified the layout text, chunk coverage and inherited overlap evidence in the read view.")

## Step 1 results

Report both the complete parent inventory and the subset examined by this rule. The updated
chunk totals use measured deltas from changed parents and the existing Notebook 04 plans
for the rest of the population.

In [ ]:
import pandas as pd
from IPython.display import display

counts = result_audit["counts"]
display(
    pd.DataFrame(
        [
            ("Formal parent records in the input", result_audit["formal_records"]),
            (
                "Parents with a recorded Notebook 03 edit examined",
                counts["records_with_03_edits_examined"],
            ),
            ("Parents changed by Step 1", counts["edited_records"]),
            ("Extra newline runs repaired", counts["repaired_runs"]),
            ("Repairs at document boundaries", counts["boundary_runs_repaired"]),
            ("Net code points removed", counts["net_codepoints_removed"]),
            ("Remaining extra newline runs in the rule scope", counts["remaining_runs_in_scope"]),
            ("Parents flagged for Step 1 review", counts["review_records"]),
        ],
        columns=["Measure", "Records or occurrences"],
    )
)
display(
    pd.DataFrame(result_audit["by_split"]).rename(
        columns={
            "ml_split": "Inherited split",
            "examined": "Parents examined by the rule",
            "edited": "Parents edited",
        }
    )
)
display(
    pd.DataFrame(result_audit["chunk_summary"]).rename(
        columns={
            "target": "Chunk target (code points)",
            "replanned_records": "Parents replanned",
            "records_with_chunk_count_change": "Parents with a different chunk count",
            "chunk_count_delta": "Change in total chunks",
            "input_chunks": "Notebook 04 chunks",
            "output_chunks": "Chunks after the layout step",
        }
    )
)

## Step 2 · Audit residual markup and normalize two verified patterns

Scan all current bodies for protocol markers, character references, angle-bracket markup,
BBCode-like strings and nonbreaking spaces. These are candidate shapes. For example, a
protocol marker may be followed by a broken path or occur in a code example. A line-leading
`&gt;` may indicate a quote and angle-bracket actions can carry tone in forums. Nonbreaking
spaces are also part of literary formatting.

Normalize complete, empty TinyMCE selection bookmarks with the observed escaping and
fixed attribute signature in publication formats. The editing record retains both
representations. Convert the specific Music Feeds Soundwave caption break into a newline,
preserving the caption text. Retain other candidate occurrences for review, including
URLs, general HTML and BBCode. The code records which patterns remain by source and format.

The residual snapshot is published beside the Step 1 snapshot using its directory name
with the suffix `_residual_v1`. The record ledger uses offsets in the Step 1 text.
Prior chunk plans must replay for every edit. Check new body hashes for exact matches
across inherited splits and retain any evidence for later review.

In [ ]:
RESIDUAL_PATTERNS = {
    "url_scheme": r"(?i)\b(?:https?|ftp)://",
    "escaped_entity": r"&(?:[A-Za-z][A-Za-z0-9]{1,31}|#[xX][0-9a-fA-F]{1,6}|#[0-9]{1,7});",
    "angle_markup_like": r"</?[A-Za-z][A-Za-z0-9:_-]*(?=[\s/>])[^<>]{0,500}>",
    "bbcode_like": (
        r"(?i)\[/?(?:quote|code|pre|url|img|b|i|u|size|color|spoiler|list|media)"
        r"(?:=[^\r\n\]]{1,160})?\]"
    ),
    "nonbreaking_space": r"[\u00a0\u202f]",
}
RESIDUAL_MATCHERS = {kind: re.compile(pattern) for kind, pattern in RESIDUAL_PATTERNS.items()}
RESIDUAL_PREFILTER = (
    "(?U)(?:" + "|".join(f"(?:{pattern})" for pattern in RESIDUAL_PATTERNS.values()) + ")"
)
EDITING_BOOKMARK = re.compile(
    r"&(?P<encoding>amp;)?lt;span (?:"
    r'data-mce-type="bookmark" style="display: inline-block; width: 0px; '
    r'overflow: hidden; line-height: 0;"|'
    r'style="display: inline-block; width: 0px; overflow: hidden; line-height: 0;" '
    r'data-mce-type="bookmark") class="mce_SELRES_start"&(?:amp;)?gt;'
    r"[ \t\ufeff\u00a0]{0,16}&(?:amp;)?lt;/span&(?:amp;)?gt;"
)
CAPTION_BREAK = re.compile(
    r"(?m)^[ \t]*[^\r\n<>`]{1,105}?(?P<tag><br />)"
    r"(?=Soundwave (?:19|20)\d{2}, [A-Za-z][A-Za-z -]{1,28}[ \t]*(?:\n|$))"
)
VERBATIM_MARKERS = re.compile(r"(?i)`|</?(?:pre|code)\b|\[/?(?:pre|code)(?=[=\]])")
NON_PROSE_FORMATS = {"Literary", "Transcript", "Unresolved", "Other"}
EDITOR_RECORDED_FORMATS = {"Article", "Blog", "News", "Community publication", "Newsletter"}
EDIT_RULE_NAMES = {
    "escaped_empty_editor_bookmark",
    "musicfeeds_soundwave_caption_break",
}


def propose_residual_edits(text, format_name, host):
    if format_name in NON_PROSE_FORMATS or VERBATIM_MARKERS.search(text):
        return []
    edits = []
    if format_name in EDITOR_RECORDED_FORMATS:
        for match in EDITING_BOOKMARK.finditer(text):
            start, end = match.span()
            fragment = match.group()
            # Use consistent escaping throughout the complete marker.
            expected = "amp;" if match["encoding"] else ""
            if any(prefix != expected for prefix in re.findall(r"&(amp;)?(?:lt|gt);", fragment)):
                continue
            # Retain explicitly quoted examples.
            if text[start - 1 : start] in {'"', "'", "“", "‘"}:
                continue
            if text[end : end + 1] in {'"', "'", "”", "’"}:
                continue
            edits.append((start, end, "escaped_empty_editor_bookmark", fragment, " "))
    if format_name == "Article" and host == "musicfeeds.com.au":
        for match in CAPTION_BREAK.finditer(text):
            start, end = match.span("tag")
            edits.append((start, end, "musicfeeds_soundwave_caption_break", "<br />", "\n"))
    return sorted(edits)


def residual_counts(text):
    counts = {}
    for kind, pattern in RESIDUAL_MATCHERS.items():
        count = sum(1 for _ in pattern.finditer(text))
        if count:
            counts[kind] = count
    return counts


def audit_residual_markup(text, input_sha, format_name, host):
    if text_sha(text) != input_sha:
        raise ValueError("Residual audit body does not match its Step 1 input hash.")
    before = residual_counts(text)
    edits = propose_residual_edits(text, format_name, host)
    if not edits:
        return before, before, [], None, input_sha, len(text), None, None
    corrected = replay_edits(text, edits)
    if not corrected.strip() or propose_residual_edits(corrected, format_name, host):
        raise ValueError("An edit emptied the parent or was not idempotent.")
    after = residual_counts(corrected)
    if any(count > before.get(kind, 0) for kind, count in after.items()):
        raise ValueError("A residual edit introduced a new watched marker.")
    spans_before = paragraph_spans(text)
    spans_after = paragraph_spans(corrected)
    revised = [
        (
            target,
            paragraph_chunk_ends(len(text), spans_before, target),
            paragraph_chunk_ends(len(corrected), spans_after, target),
        )
        for target in targets
    ]
    for _, _, ends in revised:
        if (
            not ends
            or ends[-1] != len(corrected)
            or any(lower >= upper for lower, upper in zip([0, *ends[:-1]], ends, strict=True))
        ):
            raise ValueError("The revised residual markup plan does not cover its parent.")
    return (
        before,
        after,
        edits,
        corrected,
        text_sha(corrected),
        len(corrected),
        spans_after,
        revised,
    )

In [ ]:
bookmark_tag = (
    '&lt;span data-mce-type="bookmark" style="display: inline-block; width: 0px; '
    'overflow: hidden; line-height: 0;" class="mce_SELRES_start"&gt; &lt;/span&gt;'
)
double_bookmark = bookmark_tag.replace("&", "&amp;")
caption = "Stage photo<br />Soundwave 2014, Perth\nPhoto by the author."
control_cases = [
    ("Not afraid" + bookmark_tag + "of 40°C 👩🏽‍🚒.", "Article", "example.com", 1),
    (double_bookmark + "Still worried.", "Blog", "example.com", 1),
    ("It is " + bookmark_tag.replace("; &lt;", "; I am worried&lt;"), "Blog", "example.com", 0),
    ("Documented as “" + bookmark_tag + "” here.", "Article", "example.com", 0),
    ("`" + bookmark_tag + "`", "Article", "example.com", 0),
    (bookmark_tag + "Words", "Forum", "example.com", 0),
    (caption, "Article", "musicfeeds.com.au", 1),
    (caption, "Article", "example.com", 0),
    (caption, "Forum", "musicfeeds.com.au", 0),
    ("It is &gt; 40°C. <sigh> I worry. https://   /extreme-heat", "Forum", "example.com", 0),
    ("[i] I think &amp;#3232; [quote]never afraid[/quote]", "Literary", "example.com", 0),
    ("&lt;span&gt;Don't remove me&lt;/span&gt;", "Article", "example.com", 0),
    ("Text <br /> arbitrary line break", "Article", "musicfeeds.com.au", 0),
    ("[code]" + caption + "[/code]", "Article", "musicfeeds.com.au", 0),
]
for raw, fmt, host, expected in control_cases:
    observed = audit_residual_markup(raw, text_sha(raw), fmt, host)
    assert len(observed[2]) == expected
    assert (observed[3] is not None) == bool(expected)
    assert text_sha(observed[3] if expected else raw) == observed[4]
assert audit_residual_markup(caption, text_sha(caption), "Article", "musicfeeds.com.au")[3] == (
    "Stage photo\nSoundwave 2014, Perth\nPhoto by the author."
)
assert (
    replay_edits(
        control_cases[0][0],
        propose_residual_edits(*control_cases[0][:3]),
    )
    == "Not afraid of 40°C 👩🏽‍🚒."
)
print(f"{len(control_cases)} markup edit and preservation cases passed.")

In [ ]:
RESIDUAL_OUTPUT = OUTPUT.with_name(f"{OUTPUT.name}_residual_v1")
residual_cell = next(cell for cell in notebook.cells if cell.id == "05-residual-policy")
executed_policy = None
for source in reversed(kernel.history_manager.input_hist_raw):
    try:
        parsed = ast.parse(source)
    except SyntaxError:
        continue
    if any(
        isinstance(item, ast.FunctionDef) and item.name == "audit_residual_markup"
        for item in parsed.body
    ):
        executed_policy = canonical(source)
        break
if executed_policy != canonical(residual_cell.source):
    raise ValueError("Save Notebook 05 and rerun its residual markup policy before publication.")

residual_contract = {
    "schema_version": 1,
    "layout_manifest_sha256": file_sha(OUTPUT / "MANIFEST.json"),
    "residual_policy_sha256": text_sha(executed_policy),
    "chunk_targets": list(targets),
}
source_keys = ("release_row", "native_identity", "structure_sha256")
first_hashes = first_rows.select(
    "release_row",
    "native_identity",
    F.col("input_sha256").alias("structure_sha256"),
    F.col("clean_sha256").alias("first_pass_sha256"),
    F.col("plans").alias("first_pass_plans"),
)
layout_hashes = layout_evidence.where(F.size("layout_edits") > 0).select(
    *source_keys,
    F.col("input_sha256").alias("first_pass_sha256"),
    F.col("output_sha256").alias("layout_sha256"),
    F.col("clean_text_override").alias("layout_body"),
    F.transform(
        "replanned",
        lambda plan: F.struct(plan.target.alias("target"), plan.output_ends.alias("ends")),
    ).alias("layout_plans"),
)
evidence_schema = T.StructType.fromDDL("""
    before_counts MAP<STRING,INT>, after_counts MAP<STRING,INT>,
    residual_edits ARRAY<STRUCT<start: INT, end: INT, rule: STRING,
                                before: STRING, replacement: STRING>>,
    clean_text_override STRING, output_sha256 STRING, output_chars INT,
    output_paragraphs ARRAY<STRUCT<start: INT, end: INT>>,
    replanned ARRAY<STRUCT<target: INT, input_ends: ARRAY<INT>, output_ends: ARRAY<INT>>>
""")

if RESIDUAL_OUTPUT.is_dir():
    residual_manifest, residual_audit, _ = read_snapshot(
        RESIDUAL_OUTPUT, "AUDIT.json", "audit_sha256"
    )
    if any(residual_manifest.get(key) != value for key, value in residual_contract.items()):
        raise ValueError("Use a new FOT05_OUTPUT for a changed residual policy or layout snapshot.")
    verify_parts(RESIDUAL_OUTPUT, residual_manifest["tables"])
    print(f"Reusing verified residual snapshot {RESIDUAL_OUTPUT.relative_to(ROOT)}.")
else:
    start_time = time.monotonic()
    first_overrides = first_rows.where(F.col("clean_text_override").isNotNull()).select(
        "release_row",
        "native_identity",
        F.col("input_sha256").alias("structure_sha256"),
        F.col("clean_text_override").alias("first_body"),
        F.col("clean_sha256").alias("first_text_sha256"),
    )
    layout_overrides = layout_hashes.select(*source_keys, "layout_body", "layout_sha256")
    current = (
        structure_rows.select(
            *keys, F.col("derived_sha256").alias("structure_sha256"), "derived_text"
        )
        .join(F.broadcast(first_overrides), list(source_keys), "left")
        .join(F.broadcast(layout_overrides), list(source_keys), "left")
        .select(
            *source_keys,
            "ml_split",
            "format",
            "source_host",
            F.coalesce("layout_body", "first_body", "derived_text").alias("layout_step_body"),
            F.coalesce("layout_sha256", "first_text_sha256", "structure_sha256").alias(
                "input_sha256"
            ),
        )
    )
    scan = F.udf(audit_residual_markup, evidence_schema, useArrow=False)
    with tempfile.TemporaryDirectory(prefix=".formal-residual-", dir=output_root) as temp:
        staged = Path(temp) / "snapshot"
        (
            current.where(F.col("layout_step_body").rlike(RESIDUAL_PREFILTER))
            .select(
                *source_keys,
                "ml_split",
                "format",
                "source_host",
                "input_sha256",
                F.length("layout_step_body").alias("input_chars"),
                scan("layout_step_body", "input_sha256", "format", "source_host").alias("review"),
            )
            .select("*", "review.*")
            .drop("review")
            .where((F.size("before_counts") > 0) | (F.size("residual_edits") > 0))
            .write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged / "records"))
        )
        evidence = spark.read.parquet(str(staged / "records"))
        edited_rows = evidence.where(F.size("residual_edits") > 0)
        checks = (
            evidence.agg(
                F.count("*").alias("candidate_records"),
                F.countDistinct("release_row").alias("unique_release_rows"),
                F.count_if(F.size("residual_edits") > 0).alias("edited_records"),
                F.count_if(F.size("after_counts") > 0).alias("records_with_retained_markers"),
                F.sum(F.size("residual_edits")).alias("edits"),
                F.sum(F.col("input_chars") - F.col("output_chars")).alias("net_codepoints_removed"),
                F.count_if(
                    (F.size("residual_edits") > 0) != F.col("clean_text_override").isNotNull()
                ).alias("bad_override"),
            )
            .first()
            .asDict()
        )
        if (
            checks["candidate_records"] != checks["unique_release_rows"]
            or checks["bad_override"]
            or checks["candidate_records"] > population
        ):
            raise ValueError("Residual evidence cardinality or override contract failed.")

        observed_plans = F.transform(
            "replanned",
            lambda plan: F.struct(plan.target.alias("target"), plan.input_ends.alias("ends")),
        )
        replay_check = (
            first_hashes.join(
                F.broadcast(
                    edited_rows.select(
                        *source_keys, "input_sha256", observed_plans.alias("replayed_plans")
                    )
                ),
                list(source_keys),
                "inner",
            )
            .join(
                F.broadcast(
                    layout_hashes.select(
                        *source_keys, "first_pass_sha256", "layout_sha256", "layout_plans"
                    )
                ),
                [*source_keys, "first_pass_sha256"],
                "left",
            )
            .agg(
                F.count("*").alias("replayed_records"),
                F.count_if(
                    ~F.col("input_sha256").eqNullSafe(
                        F.coalesce("layout_sha256", "first_pass_sha256")
                    )
                    | ~F.col("replayed_plans").eqNullSafe(
                        F.coalesce("layout_plans", "first_pass_plans")
                    )
                ).alias("replay_errors"),
            )
            .first()
            .asDict()
        )
        if replay_check != {"replayed_records": checks["edited_records"], "replay_errors": 0}:
            raise ValueError(f"The inherited plans or body hashes did not replay: {replay_check}")

        markers = evidence.select(
            "format",
            "source_host",
            "after_counts",
            F.explode("before_counts").alias("marker", "before"),
        ).withColumn("after", F.coalesce(F.element_at("after_counts", F.col("marker")), F.lit(0)))
        marker_metrics = (
            F.count("*").alias("records_before"),
            F.sum("before").alias("occurrences_before"),
            F.count_if(F.col("after") > 0).alias("records_after"),
            F.sum("after").alias("occurrences_after"),
        )
        totals = [
            row.asDict()
            for row in markers.groupBy("marker").agg(*marker_metrics).orderBy("marker").collect()
        ]
        (
            markers.groupBy("marker", "format", "source_host")
            .agg(*marker_metrics)
            .write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged / "marker_by_source_format"))
        )
        by_rule = [
            row.asDict()
            for row in edited_rows.select(
                "release_row", "format", "source_host", F.explode("residual_edits").alias("edit")
            )
            .groupBy(F.col("edit.rule").alias("rule"), "format", "source_host")
            .agg(F.count("*").alias("edits"), F.countDistinct("release_row").alias("records"))
            .orderBy("rule", "format", "source_host")
            .collect()
        ]
        chunk_deltas = [
            row.asDict()
            for row in edited_rows.select(F.explode("replanned").alias("plan"))
            .select("plan.*")
            .groupBy("target")
            .agg(
                F.count("*").alias("replanned_records"),
                F.sum(F.size("output_ends") - F.size("input_ends")).alias("chunk_count_delta"),
            )
            .orderBy("target")
            .collect()
        ]
        if any(row["replanned_records"] != checks["edited_records"] for row in chunk_deltas):
            raise ValueError("Each edited parent must have a revised plan at both targets.")

        # Check the new body hashes against the full roster without reading noncandidate text.
        step1_hashes = (
            first_hashes.drop("first_pass_plans")
            .join(
                F.broadcast(
                    layout_hashes.select(*source_keys, "first_pass_sha256", "layout_sha256")
                ),
                [*source_keys, "first_pass_sha256"],
                "left",
            )
            .select(
                *source_keys, F.coalesce("layout_sha256", "first_pass_sha256").alias("prior_sha")
            )
        )
        changed_hashes = edited_rows.select(
            *source_keys, "ml_split", "input_sha256", F.col("output_sha256").alias("final_sha")
        )
        roster_splits = structure_rows.select(
            "release_row",
            "native_identity",
            F.col("derived_sha256").alias("structure_sha256"),
            "ml_split",
        )
        final_hashes = (
            step1_hashes.join(
                F.broadcast(changed_hashes.drop("ml_split", "input_sha256")),
                list(source_keys),
                "left",
            )
            .join(roster_splits, list(source_keys), "inner")
            .select(
                F.col("release_row").alias("other_row"),
                F.col("ml_split").alias("other_split"),
                "prior_sha",
                F.coalesce("final_sha", "prior_sha").alias("final_sha"),
            )
        )
        cross_split = (
            final_hashes.join(F.broadcast(changed_hashes), "final_sha", "inner")
            .where(
                (F.col("other_split") != F.col("ml_split"))
                & (F.col("prior_sha") != F.col("input_sha256"))
            )
            .select(
                F.least("other_row", "release_row").alias("row_a"),
                F.greatest("other_row", "release_row").alias("row_b"),
                "final_sha",
            )
            .distinct()
        )
        cross_split.write.mode("errorifexists").parquet(str(staged / "new_exact_cross_split"))
        tables = {
            name: [
                {
                    "path": path.relative_to(staged).as_posix(),
                    "sha256": file_sha(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted((staged / name).glob("*.parquet"))
            ]
            for name in ("records", "marker_by_source_format", "new_exact_cross_split")
        }

        residual_audit = {
            "formal_records_scanned": population,
            "split_counts": split_counts,
            **checks,
            **replay_check,
            "marker_totals": totals,
            "by_rule_source_format": by_rule,
            "chunk_deltas": chunk_deltas,
            "new_exact_cross_split_pairs": sum(
                row["rows"] for row in tables["new_exact_cross_split"]
            ),
            "elapsed_seconds": round(time.monotonic() - start_time, 1),
        }
        audit_path = staged / "AUDIT.json"
        audit_path.write_text(json.dumps(residual_audit, indent=2) + "\n")
        residual_manifest = {
            **residual_contract,
            "spark_master": spark.sparkContext.master,
            "audit_sha256": file_sha(audit_path),
            "tables": tables,
            "edit_coordinates": "Half-open Unicode code point offsets in the Step 1 text",
        }
        (staged / "MANIFEST.json").write_text(json.dumps(residual_manifest, indent=2) + "\n")
        verify_parts(staged, tables)
        if file_sha(OUTPUT / "MANIFEST.json") != residual_contract["layout_manifest_sha256"]:
            raise ValueError("The layout snapshot changed during the residual scan.")
        if RESIDUAL_OUTPUT.exists():
            raise FileExistsError("The residual destination appeared during publication.")
        staged.rename(RESIDUAL_OUTPUT)
        print(f"Published the residual snapshot at {RESIDUAL_OUTPUT.relative_to(ROOT)}.")

### Residual audit results

Counts describe detected marker occurrences and parent records. A parent can contribute to
several categories. `residual_marker_by_source_format` holds detailed grouped counts.
`refined_text_records` resolves both Notebook 05 steps in order while carrying forward the
record identity, split and overlap evidence.

In [ ]:
residual_evidence = spark.read.parquet(str(RESIDUAL_OUTPUT / "records"))
residual_marker_by_source_format = spark.read.parquet(
    str(RESIDUAL_OUTPUT / "marker_by_source_format")
)
residual_key = ["release_row", "native_identity", "clean_sha256"]
refinement = residual_evidence.select(
    "release_row",
    "native_identity",
    F.col("input_sha256").alias("clean_sha256"),
    "before_counts",
    "after_counts",
    "residual_edits",
    F.col("output_paragraphs").alias("residual_paragraphs"),
    "output_chars",
    F.col("clean_text_override").alias("refinement_body"),
    F.col("output_sha256").alias("refinement_sha256"),
    F.transform(
        "replanned",
        lambda plan: F.struct(plan.target.alias("target"), plan.output_ends.alias("ends")),
    ).alias("refinement_plans"),
)


def residual_text_view(layout, refined):
    return (
        layout.join(F.broadcast(refined), residual_key, "left")
        .withColumns(
            {
                "layout_step_sha256": F.col("clean_sha256"),
                "clean_sha256": F.coalesce("refinement_sha256", "clean_sha256"),
                "clean_text": F.coalesce("refinement_body", "clean_text"),
                "clean_chars": F.coalesce("output_chars", "clean_chars"),
                "plans": F.coalesce("refinement_plans", "plans"),
            }
        )
        .drop("refinement_body", "refinement_sha256", "refinement_plans", "output_chars")
    )


refined_text_records = residual_text_view(layout_text_records, refinement)
control = (
    residual_evidence.where(F.size("residual_edits") > 0)
    .orderBy("release_row")
    .select("release_row", "output_sha256")
    .first()
)
if control is not None:
    narrow = F.col("release_row") == control.release_row
    probe = (
        residual_text_view(
            layout_text_view(
                layout_record_view(
                    first_rows.where(narrow),
                    layout_evidence.where(narrow),
                    overlap_rows.where(narrow),
                ),
                F.broadcast(structure_rows.where(narrow)),
            ),
            refinement.where(narrow),
        )
        .select("clean_text", "clean_sha256", "clean_chars", "plans")
        .collect()
    )
    assert len(probe) == 1
    assert text_sha(probe[0].clean_text) == probe[0].clean_sha256 == control.output_sha256
    assert len(probe[0].clean_text) == probe[0].clean_chars
    assert [plan.ends[-1] for plan in probe[0].plans] == [probe[0].clean_chars] * len(targets)

display(pd.DataFrame(residual_audit["marker_totals"]))
rule_summary = pd.DataFrame(residual_audit["by_rule_source_format"])
if not rule_summary.empty:
    display(rule_summary.groupby("rule", as_index=False)[["records", "edits"]].sum())
display(pd.DataFrame(residual_audit["chunk_deltas"]))
print(
    f"{residual_audit['formal_records_scanned']:,} parents scanned. "
    f"{residual_audit['candidate_records']:,} with watched markers. "
    f"{residual_audit['edited_records']:,} parents edited. "
    f"{residual_audit['new_exact_cross_split_pairs']:,} new exact cross-split body pairs."
)

## Publish the verified handoff for chunk evaluation

Publish the complete versioned parent table for Notebook 06. Check the final text hashes,
paragraph offsets and both baseline plans across all formal parents. Verify that each edit
step introduced zero new cross-split exact body pairs. Retain all known cross-split overlap
evidence and carry it forward as a diagnostic flag. Near-duplicate and containment coverage follows the scope of the bound Notebook 03 audit.
All 139 parents edited in Step 2 had different five-word shingle sets in an independent replay.
The incremental near-duplicate audit below compares these parents with both full rosters.

Retain residual URL schemes, entities, markup, nonbreaking spaces and boundary whitespace
with their review counts. These shapes can include authored expressions. Chunk budgets in
this snapshot use Unicode code points. Notebook 06 reads the materialized snapshot and its
machine-readable closure receipt.

In [ ]:
handoff = OUTPUT.with_name(f"{OUTPUT.name}_handoff_v1")
quality_path = FIRST_PASS.with_name(f"{FIRST_PASS.name}_quality_v1.json")
quality = json.loads(quality_path.read_text())
if (
    quality["first_pass_manifest_sha256"] != first_sha
    or quality["structure_manifest_sha256"] != structure_sha
    or quality["overlap_manifest_sha256"] != overlap_sha
    or quality["formal_records"] != population
):
    raise ValueError("Run Notebook 04 to bind the first pass to the expected overlap snapshot.")

handoff_sources = {
    "structure_sha256": structure_sha,
    "first_pass_sha256": first_sha,
    "first_pass_quality_sha256": file_sha(quality_path),
    "overlap_sha256": overlap_sha,
    "layout_sha256": file_sha(OUTPUT / "MANIFEST.json"),
    "residual_sha256": file_sha(RESIDUAL_OUTPUT / "MANIFEST.json"),
}
handoff_contract = {
    "schema_version": 1,
    "input_hashes": handoff_sources,
    "targets_codepoints": list(targets),
    "coordinate_system": "Zero-based half-open Unicode code point ranges",
}


def exact_pairs_introduced(records, before_hash, after_hash):
    hashes = records.select("release_row", "ml_split", before_hash, after_hash)
    modified = hashes.where(F.col(before_hash) != F.col(after_hash)).select(
        F.col("release_row").alias("changed_row"),
        F.col("ml_split").alias("changed_split"),
        F.col(before_hash).alias("changed_prior_sha"),
        after_hash,
    )
    return (
        hashes.join(F.broadcast(modified), after_hash, "inner")
        .where(
            (F.col("ml_split") != F.col("changed_split"))
            & (F.col(before_hash) != F.col("changed_prior_sha"))
        )
        .select(
            F.least("release_row", "changed_row").alias("row_a"),
            F.greatest("release_row", "changed_row").alias("row_b"),
        )
        .distinct()
        .count()
    )


if handoff.exists():
    handoff_manifest, closure, _ = read_snapshot(handoff, "05_CLOSURE.json", "audit_sha256")
    if any(handoff_manifest.get(key) != value for key, value in handoff_contract.items()):
        raise ValueError("Use a new layout version with the changed handoff inputs.")
    verify_parts(handoff, handoff_manifest["tables"])
    print(f"Reusing verified handoff {handoff.relative_to(ROOT)}.")
else:
    start = time.monotonic()
    metadata_parts = {"metadata": structure_manifest["tables"]["metadata"]["files"]}
    verify_parts(STRUCTURE, metadata_parts)
    metadata = spark.read.parquet(str(STRUCTURE / "metadata")).select(
        "release_row",
        "native_identity",
        "source_id",
        "publication_year",
        "publication_precision",
        "publication_date_start",
        "publication_date_end",
        "publication_timestamp_utc",
    )
    empty_reasons = F.array().cast("array<string>")
    paragraph_ranges = F.udf(
        lambda text: paragraph_spans(text) if text is not None else None,
        T.ArrayType(T.StructType.fromDDL("start INT, end INT")),
        useArrow=False,
    )
    first_pass_only = (
        F.col("layout_paragraphs").isNull()
        & F.col("residual_paragraphs").isNull()
        & (F.col("first_pass_sha256") != F.col("structure_sha256"))
    )
    output_records = refined_text_records.join(
        metadata, ["release_row", "native_identity"], "inner"
    ).select(
        *keys,
        "source_id",
        "publication_year",
        "publication_precision",
        "publication_date_start",
        "publication_date_end",
        "publication_timestamp_utc",
        "structure_sha256",
        "first_pass_sha256",
        "layout_step_sha256",
        "clean_text",
        "clean_sha256",
        "clean_chars",
        "plans",
        F.coalesce(
            "residual_paragraphs",
            "layout_paragraphs",
            paragraph_ranges(F.when(first_pass_only, F.col("clean_text"))),
            "structure_paragraphs",
        ).alias("paragraphs"),
        "structure_review_reasons",
        F.coalesce("review_flags", empty_reasons).alias("first_pass_review_reasons"),
        "layout_status",
        "layout_noop_reason",
        F.coalesce("layout_review_flags", empty_reasons).alias("layout_review_reasons"),
        F.col("before_counts").isNotNull().alias("residual_scanned_marker_candidate"),
        F.coalesce(F.size("residual_edits"), F.lit(0)).alias("residual_edit_count"),
        F.coalesce("after_counts", F.create_map().cast("map<string,int>")).alias(
            "retained_marker_counts"
        ),
        "overlap_status",
        F.coalesce("overlap_rules", empty_reasons).alias("overlap_rules"),
        F.coalesce("overlap_pair_ids", empty_reasons).alias("overlap_pair_ids"),
        F.col("overlap_status")
        .isin("direct_evidence", "related_release_group")
        .alias("cross_split_review_required"),
    )
    with tempfile.TemporaryDirectory(prefix=".formal-handoff-", dir=output_root) as temp:
        staged_handoff = Path(temp) / "snapshot"
        (
            output_records.write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged_handoff / "records"))
        )
        full = spark.read.parquet(str(staged_handoff / "records"))
        two_targets = F.array(*[F.lit(target) for target in targets])
        valid_plans = F.transform("plans", lambda plan: plan.target) == two_targets
        valid_plans &= F.forall(
            "plans",
            lambda plan: (
                (F.size(plan.ends) > 0)
                & (F.try_element_at(plan.ends, F.lit(-1)) == F.col("clean_chars"))
                & F.forall(
                    F.zip_with(
                        plan.ends,
                        F.slice(F.concat(F.array(F.lit(0)), plan.ends), 1, F.size(plan.ends)),
                        lambda end, start: end > start,
                    ),
                    lambda ordered: ordered,
                )
            ),
        )
        valid_paragraphs = (F.size("paragraphs") > 0) & F.forall(
            F.zip_with(
                "paragraphs",
                F.slice(
                    F.concat(F.array(F.lit(0)), F.transform("paragraphs", lambda p: p.end)),
                    1,
                    F.size("paragraphs"),
                ),
                lambda paragraph, previous_end: (
                    (paragraph.start >= previous_end)
                    & (paragraph.end > paragraph.start)
                    & (paragraph.end <= F.col("clean_chars"))
                ),
            ),
            lambda valid: valid,
        )
        tests = (
            full.agg(
                F.count("*").alias("parent_rows"),
                F.countDistinct("release_row").alias("unique_release_rows"),
                F.countDistinct("native_identity").alias("unique_native_identities"),
                F.count_if(
                    F.sha2("clean_text", 256).eqNullSafe(F.col("clean_sha256"))
                    & F.col("clean_text").isNotNull()
                    & (F.length("clean_text") == F.col("clean_chars"))
                ).alias("verified_body_hashes"),
                F.count_if(F.coalesce(valid_plans, F.lit(False))).alias("parents_with_valid_plans"),
                F.count_if(F.coalesce(valid_paragraphs, F.lit(False))).alias(
                    "parents_with_valid_paragraphs"
                ),
                F.count_if(F.col("clean_text").startswith("\n")).alias("leading_newline"),
                F.count_if(F.col("clean_text").endswith("\n")).alias("trailing_newline"),
                F.count_if(F.instr("clean_text", "\n\n\n") > 0).alias("triple_newline"),
                F.count_if(F.col("layout_status") != "outside_separator_rule_scope").alias(
                    "layout_candidates"
                ),
                F.count_if(F.col("layout_status") == "edited").alias("layout_edits"),
                F.count_if(F.col("residual_scanned_marker_candidate")).alias("residual_candidates"),
                F.count_if(F.col("residual_edit_count") > 0).alias("residual_edited_parents"),
                F.sum("residual_edit_count").alias("residual_edits"),
                F.count_if(F.size("retained_marker_counts") > 0).alias("retained_marker_records"),
                F.count_if(F.col("cross_split_review_required")).alias(
                    "cross_split_review_required"
                ),
                F.count_if(
                    F.col("structure_review_reasons").isNull() | F.col("source_id").isNull()
                ).alias("missing_metadata_or_review_fields"),
            )
            .first()
            .asDict()
        )
        required_counts = {
            "parent_rows": population,
            "unique_release_rows": population,
            "unique_native_identities": population,
            "verified_body_hashes": population,
            "parents_with_valid_plans": population,
            "parents_with_valid_paragraphs": population,
            "layout_candidates": result_audit["counts"]["records_with_03_edits_examined"],
            "layout_edits": result_audit["counts"]["edited_records"],
            "residual_candidates": residual_audit["candidate_records"],
            "residual_edited_parents": residual_audit["edited_records"],
            "residual_edits": residual_audit["edits"],
            "retained_marker_records": residual_audit["records_with_retained_markers"],
            "cross_split_review_required": overlap_audit["direct_or_related_group_records"],
            "missing_metadata_or_review_fields": 0,
        }
        if any(tests[key] != expected for key, expected in required_counts.items()):
            raise ValueError(f"The handoff failed its full roster, text or plan checks: {tests}")
        status_by_split = [
            row.asDict()
            for row in full.groupBy("ml_split", "overlap_status")
            .count()
            .orderBy("ml_split", "overlap_status")
            .collect()
        ]
        if status_by_split != quality["overlap_status_by_split"]:
            raise ValueError("The handoff's split or overlap roster disagrees with Notebook 04.")
        final_chunks = [
            row.asDict()
            for row in full.select(F.explode("plans").alias("plan"))
            .select("plan.*")
            .groupBy("target")
            .agg(F.sum(F.size("ends")).alias("chunks"))
            .orderBy("target")
            .collect()
        ]
        step2_delta = {
            row["target"]: row["chunk_count_delta"] for row in residual_audit["chunk_deltas"]
        }
        if final_chunks != [
            {
                "target": row["target"],
                "chunks": row["output_chunks"] + step2_delta.get(row["target"], 0),
            }
            for row in result_audit["chunk_summary"]
        ]:
            raise ValueError("The handoff chunk counts do not reconcile with the two edit ledgers.")
        step1_pairs = exact_pairs_introduced(full, "first_pass_sha256", "layout_step_sha256")
        step2_pairs = exact_pairs_introduced(full, "layout_step_sha256", "clean_sha256")
        if step1_pairs != 0 or step2_pairs != 0:
            raise ValueError(
                f"New cross-split exact matches require review: {step1_pairs=} {step2_pairs=}"
            )
        noop_reasons = [
            row.asDict()
            for row in full.where(F.col("layout_noop_reason").isNotNull())
            .groupBy("layout_noop_reason")
            .count()
            .orderBy("layout_noop_reason")
            .collect()
        ]
        closure = {
            "status": "verified_text_and_baseline_plans_for_chunk_evaluation",
            "checks": tests,
            "overlap_status_by_split": status_by_split,
            "overlap_audit_scope": overlap_audit["setting"],
            "final_baseline_chunks": final_chunks,
            "step1_new_cross_split_exact_pairs": step1_pairs,
            "step2_new_cross_split_exact_pairs": step2_pairs,
            "step1_noop_reasons": noop_reasons,
            "step1_codepoints_removed": result_audit["counts"]["net_codepoints_removed"],
            "step2_codepoints_removed": residual_audit["net_codepoints_removed"],
            "marker_counts": residual_audit["marker_totals"],
            "remaining_marker_policy": "Preserve source text and carry marker counts for review",
            "boundary_whitespace_policy": "Preserve residual boundary whitespace",
            "overlap_policy": "Carry evidence and route known cases to a diagnostic cohort",
            "token_budget": "To be evaluated in Notebook 06",
            "elapsed_seconds": round(time.monotonic() - start, 1),
        }
        closure_path = staged_handoff / "05_CLOSURE.json"
        closure_path.write_text(json.dumps(closure, indent=2) + "\n")
        output_tables = {
            "records": [
                {
                    "path": path.relative_to(staged_handoff).as_posix(),
                    "sha256": file_sha(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted((staged_handoff / "records").glob("*.parquet"))
            ],
        }
        handoff_manifest = {
            **handoff_contract,
            "audit_sha256": file_sha(closure_path),
            "tables": output_tables,
            "spark_version": spark.version,
            "spark_master": spark.sparkContext.master,
            "python_version": sys.version.split()[0],
        }
        (staged_handoff / "MANIFEST.json").write_text(json.dumps(handoff_manifest, indent=2) + "\n")
        verify_parts(staged_handoff, output_tables)
        if handoff.exists():
            raise FileExistsError("The handoff destination appeared during publication.")
        staged_handoff.rename(handoff)
    print(f"Published the validated handoff at {handoff.relative_to(ROOT)}.")

handoff_records = spark.read.parquet(str(handoff / "records"))
display(pd.DataFrame(closure["checks"].items(), columns=["Full corpus measure", "Count"]))
display(pd.DataFrame(closure["final_baseline_chunks"]))
display(pd.DataFrame(closure["overlap_status_by_split"]))
print(
    f"New exact cross-split pairs after Steps 1 and 2: "
    f"{closure['step1_new_cross_split_exact_pairs']} and "
    f"{closure['step2_new_cross_split_exact_pairs']}."
)

## Compare near-duplicate evidence before and after the residual edits

Use the two full formal rosters as backgrounds for the parents edited in Step 2. Reuse
Notebook 03 sketches for exact unchanged body hashes and rebuild missing sketches for other
hashes. Search both phases with at least one shared bottom-k sample and a maximum posting
bucket of 2,048 distinct bodies. Include cross-split hits from either phase in the scoring set.

Reconstruct and verify the earlier text using the residual edit ledger. Calculate exact
five-word-shingle overlap for each candidate in both phases. A match requires at least
30 shared shingles and either Jaccard similarity of at least 0.85 or containment of at least
0.90. Report new and resolved matches separately. The search has bounded candidate coverage
and concerns the 139 edited parents against the formal backgrounds. The scored evidence
stays beside the handoff and can be read independently.


In [ ]:
import zlib

from pyspark.sql import Row

WORD = re.compile(r"[^\W_]+(?:['’][^\W_]+)*", flags=re.UNICODE)
SHINGLE_WIDTH = 5
SAMPLE_SIZE = 12
MIN_SHINGLES = 30
DELTA_BUCKET_LIMIT = 2048
DELTA_MIN_SHARED_SAMPLES = 1
DELTA_JACCARD_PERCENT = 85
DELTA_CONTAINMENT_PERCENT = 90


def shingles(text):
    words = WORD.findall(text.casefold())
    columns = [words[offset:] for offset in range(SHINGLE_WIDTH)]
    return {"\x1f".join(window) for window in zip(*columns, strict=False)}


def sketch(text):
    items = shingles(text)
    support = len(items)
    if support < MIN_SHINGLES:
        return support, []
    keys = sorted({zlib.crc32(item.encode("utf-8")) for item in items})
    return support, keys[:SAMPLE_SIZE]


def recover_before_residual(text, earlier_sha, final_sha, edits):
    if text_sha(text) != final_sha:
        raise ValueError("The delta input body does not match the handoff hash.")
    result = []
    cursor = shift = 0
    for edit in edits:
        start = edit.start + shift
        end = start + len(edit.replacement)
        if start < cursor or text[start:end] != edit.replacement:
            raise ValueError("A residual edit could not be inverted.")
        result.extend((text[cursor:start], edit.before))
        shift += len(edit.replacement) - len(edit.before)
        cursor = end
    result.append(text[cursor:])
    recovered = "".join(result)
    if text_sha(recovered) != earlier_sha:
        raise ValueError("Reversing residual edits did not recover the layout-step hash.")
    return recovered


def delta_metrics(left, right):
    a, b = shingles(left), shingles(right)
    intersection, union, smaller = len(a & b), len(a | b), min(len(a), len(b))
    match = intersection >= MIN_SHINGLES and (
        intersection * 100 >= union * DELTA_JACCARD_PERCENT
        or intersection * 100 >= smaller * DELTA_CONTAINMENT_PERCENT
    )
    return intersection, union, smaller, match


example = " ".join(f"word{i}" for i in range(75))
assert delta_metrics(example, example + " more words")[-1]
assert delta_metrics(example, "unrelated phrase")[-1] is False
assert shingles(example) == shingles(example.replace(" ", "\n\n"))
assert (
    recover_before_residual(
        "ab c",
        text_sha("ab<removed>c"),
        text_sha("ab c"),
        [Row(start=2, replacement=" ", before="<removed>")],
    )
    == "ab<removed>c"
)

In [ ]:
source_03 = nbformat.read(ROOT / "notebooks/03_structure_normalization.ipynb", as_version=4)
source_03_policy = next(cell.source for cell in source_03.cells if cell.id == "53650fa8")
policy_source = next(cell.source for cell in notebook.cells if cell.id == "05-delta-policy")
sketch_symbols = {"shingles", "sketch", "WORD", "SHINGLE_WIDTH", "SAMPLE_SIZE", "MIN_SHINGLES"}


def sketch_contract(source):
    contract = {}
    for node in ast.parse(source).body:
        if isinstance(node, ast.FunctionDef) and node.name in sketch_symbols:
            contract[node.name] = ast.dump(node)
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in sketch_symbols:
                    contract[target.id] = ast.dump(node.value)
    return contract


if sketch_contract(source_03_policy) != sketch_contract(policy_source):
    raise ValueError("The copy of the Notebook 03 sketch policy did not match.")
executed_delta = None
for source in reversed(kernel.history_manager.input_hist_raw):
    if any(
        isinstance(node, ast.FunctionDef) and node.name == "delta_metrics"
        for node in ast.parse(source).body
    ):
        executed_delta = canonical(source)
        break
if executed_delta != canonical(policy_source):
    raise ValueError("Save the delta policy and rerun its code cell before publication.")

index_tables = {
    name: structure_manifest["tables"][name]["files"]
    for name in ("near_duplicate_sketches", "candidate_body_profiles")
}
delta_dir = handoff.with_name(f"{handoff.name}_near_delta_v1")
delta_identity = {
    "schema_version": 1,
    "input_handoff_manifest_sha256": file_sha(handoff / "MANIFEST.json"),
    "input_structure_manifest_sha256": structure_sha,
    "input_residual_manifest_sha256": handoff_sources["residual_sha256"],
    "policy_sha256": text_sha(canonical(policy_source)),
    "bucket_limit": DELTA_BUCKET_LIMIT,
    "minimum_shared_samples": DELTA_MIN_SHARED_SAMPLES,
    "minimum_shingles": MIN_SHINGLES,
    "jaccard_percent": DELTA_JACCARD_PERCENT,
    "containment_percent": DELTA_CONTAINMENT_PERCENT,
}
if delta_dir.exists():
    delta_manifest, delta_audit, _ = read_snapshot(delta_dir, "AUDIT.json", "audit_sha256")
    if any(delta_manifest.get(key) != value for key, value in delta_identity.items()):
        raise ValueError("Use a new delta audit version for different inputs or rules.")
    verify_parts(delta_dir, delta_manifest["tables"])
    print(f"Reusing the verified delta audit at {delta_dir.relative_to(ROOT)}.")
else:
    delta_started = time.monotonic()
    index_files = verify_parts(STRUCTURE, index_tables)
    membership_columns = ("release_row", "ml_split", "layout_step_sha256", "clean_sha256")
    residual_changes = residual_evidence.where(F.size("residual_edits") > 0).select(
        "release_row", "residual_edits"
    )
    invert = F.udf(recover_before_residual, "string", useArrow=False)
    signature = F.udf(sketch, "struct<shingle_count:int,sample_keys:array<long>>", useArrow=False)
    measure = F.udf(
        delta_metrics,
        "struct<intersection:int,union_size:int,smaller_set:int,match:boolean>",
        useArrow=False,
    )
    with tempfile.TemporaryDirectory(prefix=".formal-near-delta-", dir=output_root) as temp:
        temp_root = Path(temp)
        staged_delta = temp_root / "snapshot"

        def materialize(frame, name):
            destination = temp_root / name
            frame.write.mode("errorifexists").option("compression", "zstd").parquet(
                str(destination)
            )
            return spark.read.parquet(str(destination))

        modified = materialize(
            handoff_records.select(*membership_columns, "clean_text")
            .join(F.broadcast(residual_changes), "release_row", "inner")
            .withColumn(
                "prior_text",
                invert("clean_text", "layout_step_sha256", "clean_sha256", "residual_edits"),
            )
            .drop("residual_edits"),
            "modified",
        )
        modified_count = modified.count()
        if modified_count != residual_audit["edited_records"]:
            raise ValueError("The delta queries do not cover all residual edits.")

        state_members = materialize(
            handoff_records.select(
                "release_row",
                "ml_split",
                F.explode(
                    F.array(
                        F.struct(
                            F.lit("before").alias("phase"),
                            F.col("layout_step_sha256").alias("body_sha"),
                        ),
                        F.struct(
                            F.lit("after").alias("phase"), F.col("clean_sha256").alias("body_sha")
                        ),
                    )
                ).alias("version"),
            ).select("release_row", "ml_split", "version.*"),
            "state-members",
        )
        stages = state_members.groupBy("body_sha").agg(
            F.sort_array(F.collect_set("phase")).alias("phases")
        )
        original_index = (
            spark.read.parquet(str(STRUCTURE / "candidate_body_profiles"))
            .select(
                F.col("derived_sha256").alias("body_sha"),
                F.col("representative").alias("release_row"),
            )
            .join(spark.read.parquet(str(STRUCTURE / "near_duplicate_sketches")), "release_row")
            .drop("release_row")
        )
        attached = materialize(stages.join(original_index, "body_sha", "left"), "attached-index")
        missing = attached.where(F.col("sample_keys").isNull()).select("body_sha", "phases")
        rebuilt_index = materialize(
            handoff_records.select(F.col("clean_sha256").alias("body_sha"), "clean_text")
            .join(F.broadcast(missing.select("body_sha")), "body_sha", "inner")
            .unionByName(
                modified.select(
                    F.col("layout_step_sha256").alias("body_sha"),
                    F.col("prior_text").alias("clean_text"),
                ).join(F.broadcast(missing.select("body_sha")), "body_sha", "inner")
            )
            .dropDuplicates(["body_sha"])
            .withColumn("signature", signature("clean_text"))
            .select("body_sha", "signature.*"),
            "rebuilt-index",
        )
        index_counts = (
            attached.agg(
                F.count("*").alias("union_distinct_body_hashes"),
                F.count_if(F.col("sample_keys").isNull()).alias("missing_sketches"),
            )
            .first()
            .asDict()
        )
        if rebuilt_index.count() != index_counts["missing_sketches"]:
            raise ValueError("The current index could not recover every changed body hash.")
        current_index = attached.where(F.col("sample_keys").isNotNull()).unionByName(
            missing.join(rebuilt_index, "body_sha", "inner")
        )
        query_states = (
            modified.select(
                F.col("release_row").alias("query_row"),
                F.col("ml_split").alias("query_split"),
                F.explode(
                    F.array(
                        F.struct(
                            F.lit("before").alias("phase"),
                            F.col("layout_step_sha256").alias("query_body_sha"),
                        ),
                        F.struct(
                            F.lit("after").alias("phase"),
                            F.col("clean_sha256").alias("query_body_sha"),
                        ),
                    )
                ).alias("version"),
            )
            .select("query_row", "query_split", "version.*")
            .join(
                current_index.select(
                    F.col("body_sha").alias("query_body_sha"), "shingle_count", "sample_keys"
                ),
                "query_body_sha",
            )
        )
        queries = materialize(query_states, "query-states")
        if queries.count() != 2 * modified_count:
            raise ValueError("A query was missing from a corpus phase.")
        query_samples = queries.select(
            "query_row",
            "query_split",
            "query_body_sha",
            "phase",
            F.explode("sample_keys").alias("sample"),
        )
        postings = materialize(
            current_index.select("body_sha", "sample_keys", F.explode("phases").alias("phase"))
            .select("body_sha", "phase", F.explode("sample_keys").alias("sample"))
            .join(
                F.broadcast(query_samples.select("phase", "sample").distinct()),
                ["phase", "sample"],
                "inner",
            ),
            "queried-postings",
        )
        buckets = materialize(postings.groupBy("phase", "sample").count(), "queried-buckets")
        bucket_stats = [
            row.asDict()
            for row in buckets.groupBy("phase")
            .agg(
                F.count("*").alias("sample_buckets"),
                F.count_if(F.col("count") > DELTA_BUCKET_LIMIT).alias("above_cap"),
                F.max("count").alias("largest_bucket"),
            )
            .orderBy("phase")
            .collect()
        ]
        shared = (
            postings.join(
                F.broadcast(
                    buckets.where(F.col("count").between(2, DELTA_BUCKET_LIMIT)).select(
                        "phase", "sample"
                    )
                ),
                ["phase", "sample"],
                "inner",
            )
            .join(F.broadcast(query_samples), ["phase", "sample"], "inner")
            .where(F.col("body_sha") != F.col("query_body_sha"))
            .groupBy("phase", "query_row", "query_split", "body_sha")
            .agg(F.count("*").alias("shared_samples"))
            .where(F.col("shared_samples") >= DELTA_MIN_SHARED_SAMPLES)
        )
        candidates = materialize(
            state_members.join(F.broadcast(shared), ["phase", "body_sha"], "inner")
            .where(F.col("ml_split") != F.col("query_split"))
            .select(
                F.least("release_row", "query_row").alias("row_a"),
                F.greatest("release_row", "query_row").alias("row_b"),
                "phase",
                "shared_samples",
            )
            .groupBy("row_a", "row_b")
            .agg(
                F.max(F.when(F.col("phase") == "before", F.col("shared_samples"))).alias(
                    "before_shared_samples"
                ),
                F.max(F.when(F.col("phase") == "after", F.col("shared_samples"))).alias(
                    "after_shared_samples"
                ),
            ),
            "candidate-keys",
        )
        endpoints = (
            candidates.select(F.col("row_a").alias("release_row"))
            .union(candidates.select(F.col("row_b").alias("release_row")))
            .distinct()
        )

        bodies = materialize(
            handoff_records.select("release_row", "ml_split", "clean_text")
            .join(F.broadcast(endpoints), "release_row", "inner")
            .join(F.broadcast(modified.select("release_row", "prior_text")), "release_row", "left")
            .select(
                "release_row",
                "ml_split",
                "clean_text",
                F.coalesce("prior_text", "clean_text").alias("prior_text"),
            ),
            "candidate-bodies",
        )

        def pair_side(side):
            return bodies.select(
                F.col("release_row").alias(f"row_{side}"),
                F.col("ml_split").alias(f"split_{side}"),
                F.col("prior_text").alias(f"before_{side}"),
                F.col("clean_text").alias(f"after_{side}"),
            )

        (
            candidates.join(pair_side("a"), "row_a")
            .join(pair_side("b"), "row_b")
            .withColumns(
                {
                    "before": measure("before_a", "before_b"),
                    "after": measure("after_a", "after_b"),
                }
            )
            .drop("before_a", "before_b", "after_a", "after_b")
            .write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged_delta / "cross_split_candidates"))
        )
        delta_frame = spark.read.parquet(str(staged_delta / "cross_split_candidates"))
        delta_counts = (
            delta_frame.agg(
                F.count("*").alias("verified_pairs"),
                F.count_if(F.col("before.match")).alias("matched_before"),
                F.count_if(F.col("after.match")).alias("matched_after"),
                F.count_if(~F.col("before.match") & F.col("after.match")).alias("new_matches"),
                F.count_if(F.col("before.match") & ~F.col("after.match")).alias("resolved_matches"),
            )
            .first()
            .asDict()
        )
        if delta_counts["verified_pairs"] != candidates.count():
            raise ValueError("Some candidate endpoints were lost during exact scoring.")

        census = [
            row.asDict()
            for row in state_members.groupBy("phase")
            .agg(
                F.count("*").alias("parent_rows"),
                F.countDistinct("body_sha").alias("distinct_body_hashes"),
            )
            .orderBy("phase")
            .collect()
        ]
        if len(census) != 2 or any(row["parent_rows"] != population for row in census):
            raise ValueError("Each phase must contain the full formal roster.")
        delta_audit = {
            "changed_parent_queries": modified_count,
            "background": census,
            **index_counts,
            "rebuilt_sketches": index_counts["missing_sketches"],
            "verified_index_parts": index_files,
            "query_buckets": bucket_stats,
            **delta_counts,
            "scope": "Cross-split pairs reached in either phase by the bounded bottom-k index",
            "elapsed_seconds": round(time.monotonic() - delta_started, 1),
        }
        table = "cross_split_candidates"
        output_parts = {
            table: [
                {
                    "path": path.relative_to(staged_delta).as_posix(),
                    "sha256": file_sha(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted((staged_delta / table).glob("*.parquet"))
            ]
        }
        audit_path = staged_delta / "AUDIT.json"
        audit_path.write_text(json.dumps(delta_audit, indent=2) + "\n")
        delta_manifest = {
            **delta_identity,
            "audit_sha256": file_sha(audit_path),
            "tables": output_parts,
        }
        (staged_delta / "MANIFEST.json").write_text(json.dumps(delta_manifest, indent=2) + "\n")
        verify_parts(staged_delta, output_parts)
        if delta_dir.exists():
            raise FileExistsError("The delta audit destination appeared during publication.")
        staged_delta.rename(delta_dir)

near_delta_pairs = spark.read.parquet(str(delta_dir / "cross_split_candidates"))
display(pd.DataFrame([delta_audit["background"][0], delta_audit["background"][1]]))
display(
    pd.DataFrame(
        [
            {"Measure": key, "Count": delta_audit[key]}
            for key in (
                "changed_parent_queries",
                "rebuilt_sketches",
                "verified_pairs",
                "matched_before",
                "matched_after",
                "new_matches",
                "resolved_matches",
            )
        ]
    )
)
display(pd.DataFrame(delta_audit["query_buckets"]))

## Handoff to Notebook 06 and later analysis

Notebook 06 reads the versioned text, reports both baseline plans and publishes a full
review queue for the declared complex-source and record scope. The program proposes
boundaries. Each queued record receives review windows covering all its text, and each
reviewer can approve the proposals, select other boundary IDs or escalate. The program
validates source hashes and assembles offsets after all units are accepted. Model-specific
token limits and cross-split chunk reuse are separate checks on the adopted plan.

The coordinating Codex window assigns complete complex parents in small batches. Each
review window reads the same immutable packet file and returns a separate receipt.
The program handles routine boundaries and validates adjustments. Measure elapsed
time, agreement, evidence-span retention and escalations before expanding dispatch.
A new text or plan version requires a separately bound review run.

[LumberChunker](https://aclanthology.org/2024.findings-emnlp.377/) ·
[Codex window handoff](PREPROCESSING_WINDOW_HANDOFF.md)

### Exploratory analysis on the unlabeled corpus

Start with a sparse parent-level TF–IDF vocabulary of unigrams and bigrams. Preserve
expressions with negation, attribution, intensity and temperature. Compare trigrams with
minimum document support as a feature variant. Fit NMF on a reproducible development
cohort, then project comparable source-period cohorts into the same vocabulary and topic
basis. Report top-weighted terms, representative and counterexample passages, host
concentration, and stability across seeds and resamples. Plot standardized parent-level
topic rates by year alongside their denominators and missing-date coverage.

Use normalized TruncatedSVD scores for K-means or a capped sample/medoid hierarchy.
Use UMAP to display the structure. Read the dendrogram as similarity at a fixed
representation and linkage. Use separate period comparisons for chronology. For the
co-occurrence analysis, choose a fixed within-sentence or token window, report raw counts
and denominator alongside PMI, require minimum support and repeat on matched-source
periods. Keep the full corpus and source-balanced estimates available for comparison.

[NMF](https://pubmed.ncbi.nlm.nih.gov/10548103/) ·
[Human evaluation of topic interpretability](https://www.cs.columbia.edu/~blei/papers/ChangBoyd-GraberWangGerrishBlei2009.pdf) ·
[UMAP](https://arxiv.org/abs/1802.03426) ·
[PMI in lexical association](https://aclanthology.org/J90-1003/)

### Later representation and supervised comparisons

For a self-supervised comparison, start with the project's pinned MiniLM encoder.
Compare frozen representations with the same initialization adapted on training
passages using a SimCSE-style dropout objective. The two views of each original passage
form the positive pair. Keep passages from the same parent, thread and known duplicate or
overlap group out of one another's negative pool. Freeze the chunk policy, tokenizer and
pooling across the comparison. Evaluate held-out retrieval on human relevance judgements
using precision@k and nDCG, including risk-only, negated and attributed counterexamples.
ModernBERT with a masked language modeling objective is a later, higher-compute
domain adaptation alternative.

The supervised reference uses regularized logistic regression on word and phrase TF–IDF,
with a linear SVM comparison. Start with the proposal's development annotation batch
to refine the task and estimate reviewer agreement. Record evidence spans, the link to
temperature and the speaker or source of the emotion. Treat indeterminate cases as
review items. Keep parent and duplicate groups together. Report positive-class average
precision and precision/recall at a development-selected threshold, with errors by site,
format and period. As a later test, use LightGBM with SHAP on structured features such as
collective pronouns, future framing and reported attribution. Check interactions on held-out
data and inspect examples alongside the coefficients or SHAP values.

[SimCSE](https://aclanthology.org/2021.emnlp-main.552/) ·
[Domain-adaptive pretraining](https://aclanthology.org/2020.acl-main.740/) ·
[ModernBERT](https://arxiv.org/abs/2412.13663) ·
[Sparse text baselines](https://aclanthology.org/P12-2018/) ·
[TreeSHAP](https://doi.org/10.1038/s42256-019-0138-9)

### Evidence for the cultural hypothesis

Treat the statement that fear of rising temperatures is a learned, socially shared cultural
heritage as a research hypothesis. Test whether recurring forms persist across periods,
appear in independent sources and authors, and recur in passages with dated citations,
replies or explicit references to shared memory and reported experiences. Compare against risk-only reporting,
negation and direct physical exposure. Check event timing, source composition and
syndication. Interpret associations and feature attributions alongside contextual reading.
Learning and transmission require evidence beyond recurrence, trends or model attribution.

[Confounding of social influence in observational networks](https://pmc.ncbi.nlm.nih.gov/articles/PMC3328971/)